In [1]:
#Imports & Load Data
import pandas as pd
import numpy as np

df = pd.read_csv("week2_customer_transactions_messy.csv")

print("Shape:", df.shape)
df.head()

Shape: (11, 9)


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
0,T0001,C100,2026-01-05,120.50,EUR,card,completed,DE,2026-01-05
1,T0002,C101,2026/01/06,0.00,EUR,CARD,completed,de,2026-01-20
2,T0003,C102,06-01-2026,-35.00,USD,bank_transfer,completed,US,2026-01-07
3,T0004,NaN,2026-01-07,250.00,EUR,card,pending,FR,2026-01-08
4,T0005,C104,2026-01-07,89.99,EURO,cash,completed,DE,2026-01-09


In [ ]:
# Task 1 – Dataset Description

The dataset contains customer transaction records including transaction IDs, customer IDs, transaction dates, and transaction amounts.

It can be used for business applications such as:
- Customer behavior analysis
- Fraud detection
- Revenue tracking and reporting

In [3]:
#Issues by Dimension
issue_table = pd.DataFrame([
    ['Missing customer_id', 'Completeness', 'Impacts customer analytics'],
    ['Duplicate transaction_id', 'Uniqueness', 'May double count revenue'],
    ['Negative or invalid amount', 'Validity', 'Incorrect financial reporting'],
    ['Invalid date format', 'Consistency', 'Affects time-based analysis'],
    ['Broken relationships (missing IDs)', 'Integrity', 'Impacts joins and linking']
], columns=['Issue','Dimension','Impact'])

issue_table

,Issue,Dimension,Impact
0,Missing customer_id,Completeness,Impacts customer analytics
1,Duplicate transaction_id,Uniqueness,May double count revenue
2,Negative or invalid amount,Validity,Incorrect financial reporting
3,Invalid date format,Consistency,Affects time-based analysis
4,Broken relationships (missing IDs),Integrity,Impacts joins and linking


In [4]:
#KPI Calculations
kpis = {}

# Completeness
kpis['Completeness Rate'] = 1 - (df.isna().sum().sum() / (df.shape[0] * df.shape[1]))

# Duplication
kpis['Duplication Rate'] = df.duplicated(subset=['transaction_id']).mean()

# Error Rate
amount = pd.to_numeric(df['amount'], errors='coerce')
date_ok = pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed').notna()

kpis['Error Rate'] = (amount.isna() | (amount < 0) | ~date_ok).mean()

pd.DataFrame({'KPI': list(kpis.keys()), 'Value': list(kpis.values())})

,KPI,Value
0,Completeness Rate,0.959596
1,Duplication Rate,0.090909
2,Error Rate,0.272727


In [ ]:
#KPI Interpretation 

- Completeness Rate shows the proportion of non-missing data. Lower values indicate missing data issues.
- Duplication Rate indicates how many duplicate transaction IDs exist, which may lead to incorrect revenue calculations.
- Error Rate highlights invalid records such as negative amounts or incorrect date formats.

The Error Rate provides the strongest signal as it directly impacts data reliability.

In [6]:
#Validation Rules
rules = {
    'transaction_id_required': df['transaction_id'].isna() | (df['transaction_id'].astype(str).str.strip() == ''),
    'amount_non_negative': pd.to_numeric(df['amount'], errors='coerce') < 0,
    'transaction_date_valid': pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed').isna(),
}

pd.DataFrame({k: int(v.sum()) for k, v in rules.items()}, index=['affected_rows']).T

,affected_rows
transaction_id_required,0
amount_non_negative,1
transaction_date_valid,1


In [7]:
#Audit Summary
audit_summary = pd.DataFrame([
    ['Missing values', int(df.isna().sum().sum()), 'High', 'Impute or remove missing data'],
    ['Duplicate transaction_id', int(df.duplicated(subset=['transaction_id']).sum()), 'Medium', 'Remove duplicates'],
    ['Invalid amount/date', int((pd.to_numeric(df['amount'], errors='coerce').isna()).sum()), 'High', 'Validate and correct values']
], columns=['issue_type','affected_rows','severity','recommended_next_action'])

audit_summary

,issue_type,affected_rows,severity,recommended_next_action
0,Missing values,4,High,Impute or remove missing data
1,Duplicate transaction_id,1,Medium,Remove duplicates
2,Invalid amount/date,1,High,Validate and correct values


In [ ]:
#Task 6 – Cleaning Recommendations

- Handle missing values using imputation or removal
- Remove duplicate transaction records
- Convert and validate data types (amount, date)
- Standardize date formats across dataset

In [ ]:
#Reflection

- The Error Rate KPI gave the strongest signal as it directly impacts data usability.
- Invalid amounts and dates should be escalated first due to high severity.
- Additional metadata such as data source, schema definitions, and constraints would improve this audit.

In [9]:
#Function
def summarize_rule_violations(rule_dictionary):
    """
    Summarize affected-row counts for each validation rule.

    Parameters
    ----------
    rule_dictionary : dict[str, pd.Series]
        Dictionary where keys are rule names and values are boolean masks.

    Returns
    -------
    pd.DataFrame
        Table with one row per rule and violation counts.
    """
    return pd.DataFrame({
        'rule_name': list(rule_dictionary.keys()),
        'affected_rows': [int(mask.sum()) for mask in rule_dictionary.values()]
    }).sort_values('affected_rows', ascending=False)

summarize_rule_violations(rules)

,rule_name,affected_rows
1,amount_non_negative,1
2,transaction_date_valid,1
0,transaction_id_required,0
